# PromptedCRE — Autoresearch Market Refresh v2.0

**What this does:**
1. Searches the web for fresh industrial RE market data (Houston, SA, Texas, national)
2. Fetches actual content from key market report pages
3. Claude synthesizes findings and produces an updated `MARKET-CONTEXT.md`
4. Claude reviews each workflow file and flags anything needing improvement
5. Pushes everything to GitHub automatically

**Run this:** Every Monday morning. Takes ~5 minutes.

---
**Fill in your API keys in the cell below before running.**

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

ANTHROPIC_API_KEY = ""   # console.anthropic.com
BRAVE_API_KEY = ""       # api.search.brave.com (free tier works)
GITHUB_TOKEN = ""        # GitHub > Settings > Developer Settings > Personal Access Tokens (repo scope)
GITHUB_REPO = "dueriii/v0-premium-industrial-landing-page"
GITHUB_BRANCH = "main"

# Paths inside the repo
MARKET_CONTEXT_PATH = "repo/MARKET-CONTEXT.md"
WORKFLOW_PATHS = [
    "repo/workflows/01-intake.md",
    "repo/workflows/02-search-filters.md",
    "repo/workflows/03-property-survey.md",
    "repo/workflows/04-comparison.md",
    "repo/workflows/05-tour-prep.md",
    "repo/workflows/06-landlord-questions.md",
    "repo/workflows/07-deal-guidance.md",
    "repo/workflows/08-memo-output.md",
]
CHANGELOG_PATH = "repo/CHANGELOG.md"

# Research queries — expand these to add more markets
SEARCH_QUERIES = [
    "Houston Texas industrial real estate vacancy rates rents 2026",
    "Houston industrial market report Q4 2025",
    "San Antonio industrial real estate market 2026",
    "Schertz New Braunfels I-35 industrial lease rates 2026",
    "Texas industrial cap rates single tenant NNN 2026",
    "small bay industrial market trends 2026",
    "Houston industrial construction pipeline deliveries 2026",
    "industrial real estate reshoring manufacturing demand 2026",
]

In [ ]:
!pip install anthropic requests -q

import anthropic
import requests
import json
import base64
import re
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print(f"PromptedCRE Autoresearch — {today}")
print("=" * 50)

## Step 1 — Search for Fresh Market Data

In [ ]:
def brave_search(query, count=6, freshness="pm"):
    """Search Brave for fresh market data."""
    headers = {
        "Accept": "application/json",
        "X-Subscription-Token": BRAVE_API_KEY
    }
    params = {"q": query, "count": count, "freshness": freshness}
    resp = requests.get(
        "https://api.search.brave.com/res/v1/web/search",
        headers=headers, params=params, timeout=10
    )
    if resp.status_code != 200:
        return []
    results = resp.json().get("web", {}).get("results", [])
    return [{
        "title": r.get("title", ""),
        "url": r.get("url", ""),
        "description": r.get("description", ""),
        "published": r.get("page_age", "")
    } for r in results]


def fetch_page_text(url, max_chars=3000):
    """Try to fetch readable text from a URL."""
    try:
        resp = requests.get(url, timeout=8, headers={"User-Agent": "Mozilla/5.0"})
        if resp.status_code != 200:
            return ""
        # Strip HTML tags roughly
        text = re.sub(r'<[^>]+>', ' ', resp.text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text[:max_chars]
    except Exception:
        return ""


print("Searching for fresh market data...\n")
all_research = []

for query in SEARCH_QUERIES:
    print(f"  🔍 {query}")
    results = brave_search(query)
    for r in results[:3]:  # Top 3 per query
        all_research.append({
            "query": query,
            "title": r["title"],
            "url": r["url"],
            "snippet": r["description"],
            "published": r["published"]
        })
    print(f"     Found {len(results)} results")

print(f"\nTotal research items collected: {len(all_research)}")

## Step 2 — Load Repo Files from GitHub

In [ ]:
def github_get_file(path):
    """Fetch a file from GitHub. Returns (content, sha)."""
    url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{path}"
    headers = {
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "Accept": "application/vnd.github+json"
    }
    resp = requests.get(url, headers=headers, params={"ref": GITHUB_BRANCH})
    resp.raise_for_status()
    data = resp.json()
    content = base64.b64decode(data["content"]).decode("utf-8")
    return content, data["sha"]


def github_update_file(path, content, sha, message):
    """Update a file on GitHub."""
    url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{path}"
    headers = {
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "Accept": "application/vnd.github+json"
    }
    payload = {
        "message": message,
        "content": base64.b64encode(content.encode("utf-8")).decode("utf-8"),
        "sha": sha,
        "branch": GITHUB_BRANCH
    }
    resp = requests.put(url, headers=headers, json=payload)
    resp.raise_for_status()
    return resp.json()


print("Loading files from GitHub...")
current_context, context_sha = github_get_file(MARKET_CONTEXT_PATH)
print(f"  ✅ MARKET-CONTEXT.md ({len(current_context)} chars)")

workflow_files = {}
workflow_shas = {}
for path in WORKFLOW_PATHS:
    content, sha = github_get_file(path)
    workflow_files[path] = content
    workflow_shas[path] = sha
    name = path.split("/")[-1]
    print(f"  ✅ {name} ({len(content)} chars)")

changelog, changelog_sha = github_get_file(CHANGELOG_PATH)
print(f"  ✅ CHANGELOG.md")
print("\nAll files loaded.")

## Step 3 — Update MARKET-CONTEXT.md

In [ ]:
def format_research(research_items):
    lines = []
    for r in research_items:
        lines.append(f"\nSource: {r['title']} ({r['published']})")
        lines.append(f"URL: {r['url']}")
        lines.append(f"Query: {r['query']}")
        lines.append(f"Snippet: {r['snippet']}")
        lines.append("---")
    return "\n".join(lines)


research_digest = format_research(all_research)

market_update_prompt = f"""You are an industrial real estate market researcher specializing in Houston and San Antonio Texas.
Today is {today}.

CURRENT MARKET-CONTEXT.md:
```
{current_context}
```

FRESH RESEARCH GATHERED TODAY:
{research_digest}

TASK:
Produce an updated MARKET-CONTEXT.md that:
1. Keeps the exact same structure and headers
2. Updates numbers where fresh data supports a change (note source)
3. Adds any significant new data points not currently captured
4. Updates the Last updated date to {today}
5. Ends with a ## WHAT CHANGED section listing specific updates made (or "No material changes" if data is consistent)

Rules:
- Only update a number if you have a specific source backing it
- Flag estimated vs confirmed data with (est.) or (confirmed)
- If research found nothing new, say so clearly in WHAT CHANGED
- Keep the file concise and scannable — brokers and occupiers need to use this quickly

Output ONLY the complete updated markdown file. No preamble."""

print("Running market context update with Claude...")
response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=4096,
    messages=[{"role": "user", "content": market_update_prompt}]
)
updated_context = response.content[0].text.strip()

print(f"\nUpdated context ({len(updated_context)} chars)")
print("\nPREVIEW (first 1500 chars):")
print("-" * 50)
print(updated_context[:1500])

## Step 4 — Run Workflow Self-Improvement Loop

In [ ]:
workflow_flags = {}

# Build a combined summary of all workflow files for context
workflow_summary = "\n\n".join([
    f"### {path.split('/')[-1]}\n{content[:800]}..."
    for path, content in workflow_files.items()
])

improvement_prompt = f"""You are an industrial real estate workflow expert reviewing a set of AI workflow files.
Today is {today}.

UPDATED MARKET CONTEXT:
{updated_context[:2000]}

CURRENT WORKFLOW FILES (summaries):
{workflow_summary}

TASK — Observe, Inspect, and Flag:
Review each workflow file. For each one, determine:
1. Are there any data points that conflict with the updated market context?
2. Are there missing scenarios, edge cases, or questions that should be added?
3. Is the prompting clear enough that a non-expert user would get useful output?
4. Any Texas-specific nuances missing (incentives, local customs, market structure)?

Output format — for EACH workflow:
## [filename]
Status: [✅ Good as-is | ⚠️ Minor improvements suggested | 🔴 Needs update]
Flags: [Bullet list of specific improvements, or "None" if good as-is]

Then end with:
## PRIORITY ACTIONS
[Top 3 highest-priority improvements across all workflows, ranked by impact]

Be specific and actionable. Don't say "add more detail" — say exactly what detail to add."""

print("Running workflow self-improvement analysis...")
wf_response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=3000,
    messages=[{"role": "user", "content": improvement_prompt}]
)
workflow_analysis = wf_response.content[0].text.strip()

print("\nWORKFLOW ANALYSIS:")
print("=" * 50)
print(workflow_analysis)

## Step 5 — Push Updates to GitHub

In [ ]:
# Update MARKET-CONTEXT.md
print("Pushing MARKET-CONTEXT.md to GitHub...")
result = github_update_file(
    MARKET_CONTEXT_PATH,
    updated_context,
    context_sha,
    f"autoresearch: update MARKET-CONTEXT.md — {today}"
)
print(f"  ✅ Pushed (commit: {result['commit']['sha'][:8]})")

# Update CHANGELOG.md
new_changelog_entry = f"""
## Autoresearch — {today}

**Market context refreshed.** See WHAT CHANGED section in MARKET-CONTEXT.md.

**Workflow analysis:**
{workflow_analysis[:500]}...

---
"""
updated_changelog = changelog.replace("# CHANGELOG", "# CHANGELOG\n" + new_changelog_entry, 1)

print("Pushing CHANGELOG.md...")
result2 = github_update_file(
    CHANGELOG_PATH,
    updated_changelog,
    changelog_sha,
    f"autoresearch: log market refresh — {today}"
)
print(f"  ✅ Pushed (commit: {result2['commit']['sha'][:8]})")

print("\n" + "=" * 50)
print("RUN COMPLETE")
print("=" * 50)

## Step 6 — Your Action Items

In [ ]:
print("\n🎯 WHAT JUST HAPPENED:")
print(f"  • Searched {len(SEARCH_QUERIES)} market queries")
print(f"  • Collected {len(all_research)} research items")
print(f"  • Updated MARKET-CONTEXT.md with fresh data")
print(f"  • Analyzed all {len(WORKFLOW_PATHS)} workflow files for improvement opportunities")
print(f"  • Pushed 2 files to GitHub")

print("\n📋 WORKFLOW IMPROVEMENT PRIORITIES:")
# Extract just the PRIORITY ACTIONS section
if "## PRIORITY ACTIONS" in workflow_analysis:
    priority_section = workflow_analysis.split("## PRIORITY ACTIONS")[1].strip()
    print(priority_section)
else:
    print(workflow_analysis[-800:])

print("\n✅ Repo updated. Run again next Monday.")

---
## Optional: Auto-Schedule with GitHub Actions

To run this automatically every Monday at 7 AM CT, add this file to your repo at `.github/workflows/autoresearch.yml`:

```yaml
name: Market Autoresearch
on:
  schedule:
    - cron: '0 13 * * 1'  # Every Monday 1PM UTC = 7AM CT
  workflow_dispatch:  # Also allows manual trigger

jobs:
  refresh:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: '3.10'
      - run: pip install anthropic requests
      - run: python repo/autoresearch/market_refresh.py
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
          BRAVE_API_KEY: ${{ secrets.BRAVE_API_KEY }}
          GITHUB_TOKEN: ${{ secrets.GITHUB_TOKEN }}
```

Then add your API keys as GitHub Secrets (repo Settings → Secrets → Actions).
This runs without opening Colab — fully automated.
